# 🗄️ SQL Introductorio para Análisis de Datos

**Módulo 2** | **Sesión 4** | **Duración estimada: 1h**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-02-data-wrangling/notebooks/02_03_sql_introductorio.ipynb)

---

En esta sesión aprenderemos **SQL (Structured Query Language)**, el lenguaje universal para consultar bases de datos. No necesitas ser programador: SQL es un lenguaje declarativo donde describes **qué datos quieres**, no cómo obtenerlos. Es como hacerle preguntas a una base de datos en un lenguaje semi-natural.

## 🎯 Objetivos de Aprendizaje

| # | Competencia | Nivel | Indicador de logro |
|---|-----------|-------|--------------------|
| 1 | Comprender qué es SQL y su relación con herramientas de BI | Conceptual | Explica por qué SQL es relevante para el análisis de datos |
| 2 | Escribir consultas SELECT con filtros WHERE | Procedimental | Extrae subconjuntos de datos con condiciones específicas |
| 3 | Usar funciones de agregación y GROUP BY | Procedimental | Calcula estadísticas agrupadas (promedios, conteos) |
| 4 | Combinar tablas con JOIN | Procedimental | Une datos de dos fuentes para análisis cruzado |
| 5 | Escribir subconsultas básicas | Aplicado | Resuelve preguntas complejas anidando consultas |

## 💡 ¿Por qué es importante para tu práctica docente?

Como docente de FACES, interactuás frecuentemente con datos académicos: listas de calificaciones, registros de matrícula, indicadores de gestión. Muchos de estos datos viven en **bases de datos relacionales** que se consultan con SQL.

**Aplicaciones directas:**
- 🏫 **Gestión académica:** Consultar rendimiento estudiantil por carrera, semestre o género
- 📊 **Reportes institucionales:** Extraer datos para informes de acreditación o evaluación
- 🔧 **Power BI y herramientas BI:** Power BI, Tableau y Google Data Studio usan SQL internamente para consultar datos. Entender SQL te da control sobre lo que estas herramientas hacen "por detrás"
- 📝 **Investigación:** Acceder a bases de datos públicas (Banco Central, INE, organismos multilaterales)

> 📣 **Dato clave:** Según LinkedIn, SQL es la habilidad técnica más demandada en empleos de análisis de datos, por encima de Python y Excel.

---

## 🗄️ Sección 1: ¿Qué es SQL y por qué importa?

**SQL** (*Structured Query Language*) es el lenguaje estándar para interactuar con bases de datos relacionales. Fue creado en los años 70 en IBM y hoy es utilizado por prácticamente todas las organizaciones del mundo.

### Conceptos clave

| Concepto | Analogía con Excel | Descripción |
|----------|-------------------|------------|
| **Base de datos** | Un archivo de Excel completo | Colección organizada de datos |
| **Tabla** | Una hoja del archivo | Datos organizados en filas y columnas |
| **Fila (registro)** | Una fila de la hoja | Un registro individual (ej: un estudiante) |
| **Columna (campo)** | Una columna de la hoja | Un atributo (ej: nombre, carrera, promedio) |
| **Consulta (query)** | Un filtro o fórmula | Instrucción para extraer datos específicos |

### ¿Por qué SQL y no solo Excel?

- **Escala:** SQL maneja millones de registros sin problemas; Excel se ralentiza con más de 100,000 filas
- **Reproducibilidad:** Una consulta SQL es un código que puedes guardar, compartir y re-ejecutar
- **Integridad:** Las bases de datos relacionales tienen reglas que evitan datos inconsistentes
- **Colaboración:** Múltiples usuarios pueden consultar la misma base de datos simultáneamente

### SQL dentro de las herramientas de BI

Cuando usas Power BI y arrastras una columna para crear un gráfico, **internamente se genera una consulta SQL**. Cuando aplicas un filtro en Tableau, se traduce a una cláusula `WHERE`. Entender SQL te permite:
- Escribir consultas personalizadas en Power BI (modo DirectQuery)
- Depurar problemas cuando los datos no se muestran como esperas
- Optimizar reportes que cargan lentamente

---

## ⚙️ Sección 2: Configuración — SQLite en Python

Usaremos **SQLite**, una base de datos ligera que viene incluida con Python. No necesitas instalar nada adicional. Cargaremos nuestros archivos CSV en tablas SQLite para practicar consultas.

### Plan:
1. Importar las bibliotecas necesarias
2. Crear una base de datos en memoria
3. Cargar los CSVs como tablas

In [ ]:
# ============================================================
# Configuración inicial: bibliotecas y conexión a base de datos
# ============================================================

import sqlite3
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Crear una base de datos en memoria (no genera ningún archivo)
conexion = sqlite3.connect(':memory:')

print("✅ Conexión a base de datos SQLite creada exitosamente")
print(f"   Versión SQLite: {sqlite3.sqlite_version}")

In [ ]:
# ============================================================
# Cargar los datasets CSV como tablas en la base de datos
# ============================================================

# Cargar rendimiento académico
df_rendimiento = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')
df_rendimiento.to_sql('rendimiento', conexion, index=False, if_exists='replace')

# Cargar matrícula
df_matricula = pd.read_csv('../datasets/universidad/matricula_faces.csv')
df_matricula.to_sql('matricula', conexion, index=False, if_exists='replace')

print("✅ Tablas cargadas exitosamente:")
print(f"   - rendimiento: {len(df_rendimiento):,} registros, {len(df_rendimiento.columns)} columnas")
print(f"   - matricula:   {len(df_matricula):,} registros, {len(df_matricula.columns)} columnas")

print("\n📝 Columnas en 'rendimiento':")
print(f"   {list(df_rendimiento.columns)}")

print("\n📝 Columnas en 'matricula':")
print(f"   {list(df_matricula.columns)}")

In [ ]:
# ============================================================
# Función auxiliar para ejecutar consultas SQL y ver resultados
# ============================================================

def consulta(sql, conn=conexion):
    """
    Ejecuta una consulta SQL y devuelve el resultado como DataFrame.
    También imprime la consulta para referencia.
    """
    print(f"\033[94m-- Consulta SQL:\033[0m")
    print(f"\033[94m{sql}\033[0m")
    print()
    resultado = pd.read_sql_query(sql, conn)
    return resultado

# Probar con una consulta simple
resultado = consulta("SELECT * FROM rendimiento LIMIT 5")
print(resultado.to_string(index=False))

---

## 🔍 Sección 3: SELECT y WHERE — Seleccionar y filtrar datos

La estructura básica de una consulta SQL es:

```sql
SELECT columnas
FROM tabla
WHERE condiciones
ORDER BY columna
LIMIT n
```

| Cláusula | Función | Analogía en Excel |
|----------|---------|-------------------|
| `SELECT` | Elige qué columnas mostrar | Seleccionar columnas visibles |
| `FROM` | Indica de qué tabla leer | Elegir la hoja |
| `WHERE` | Filtra filas por condiciones | Filtro automático |
| `ORDER BY` | Ordena los resultados | Ordenar A-Z o Z-A |
| `LIMIT` | Limita cantidad de resultados | Ver solo las primeras N filas |

In [ ]:
# ============================================================
# Ejemplo 1: Seleccionar todos los estudiantes de Economía
# ============================================================

resultado = consulta("""
SELECT estudiante_id, carrera, semestre, promedio, asistencia_pct
FROM rendimiento
WHERE carrera = 'Economia'
LIMIT 10
""")

print(resultado.to_string(index=False))
print(f"\n📊 Se muestran 10 de los estudiantes de Economía")

In [ ]:
# ============================================================
# Ejemplo 2: Filtrar por promedio mayor a 15
# ============================================================
# En el sistema venezolano, la escala es de 0 a 20.
# Un promedio > 15 indica buen rendimiento académico.

resultado = consulta("""
SELECT estudiante_id, carrera, semestre, promedio, genero
FROM rendimiento
WHERE promedio > 15
ORDER BY promedio DESC
LIMIT 15
""")

print(resultado.to_string(index=False))
print(f"\n🌟 Los 15 estudiantes con mejor promedio (>15)")

In [ ]:
# ============================================================
# Ejemplo 3: Condiciones múltiples con AND / OR
# ============================================================
# Buscar estudiantes de Contaduría con beca Y que trabajan

resultado = consulta("""
SELECT estudiante_id, carrera, promedio, beca, trabaja
FROM rendimiento
WHERE carrera = 'Contaduria'
  AND beca = 'True'
  AND trabaja = 'True'
ORDER BY promedio DESC
""")

print(resultado.to_string(index=False))
print(f"\n📊 Estudiantes de Contaduría con beca que trabajan: {len(resultado)}")

In [ ]:
# ============================================================
# Ejemplo 4: Uso de OR y operadores de comparación
# ============================================================
# Estudiantes en riesgo: promedio < 10 O asistencia < 60%

resultado = consulta("""
SELECT estudiante_id, carrera, semestre, promedio, asistencia_pct
FROM rendimiento
WHERE promedio < 10
   OR asistencia_pct < 60
ORDER BY promedio ASC
LIMIT 15
""")

print(resultado.to_string(index=False))
print(f"\n⚠️ Estudiantes en riesgo académico (primeros 15)")

In [ ]:
# ============================================================
# Ejemplo 5: ORDER BY y LIMIT
# ============================================================
# Top 10 estudiantes con mejor promedio general

resultado = consulta("""
SELECT estudiante_id, carrera, semestre, promedio, 
       materias_aprobadas, materias_reprobadas
FROM rendimiento
ORDER BY promedio DESC
LIMIT 10
""")

print("🏆 Top 10 estudiantes por promedio:")
print(resultado.to_string(index=False))

### 📌 Resumen de operadores WHERE

| Operador | Significado | Ejemplo |
|----------|------------|--------|
| `=` | Igual a | `WHERE carrera = 'Economia'` |
| `>`, `<` | Mayor/menor que | `WHERE promedio > 15` |
| `>=`, `<=` | Mayor/menor o igual | `WHERE edad >= 18` |
| `<>` o `!=` | Diferente de | `WHERE carrera <> 'Contaduria'` |
| `AND` | Ambas condiciones | `WHERE promedio > 15 AND beca = 'True'` |
| `OR` | Al menos una condición | `WHERE carrera = 'Economia' OR carrera = 'Administracion'` |
| `IN` | Lista de valores | `WHERE carrera IN ('Economia', 'Contaduria')` |
| `BETWEEN` | Rango inclusivo | `WHERE promedio BETWEEN 10 AND 15` |
| `LIKE` | Patrón de texto | `WHERE carrera LIKE 'Admin%'` |

---

## 📊 Sección 4: Funciones de agregación y GROUP BY

Las funciones de agregación resumen múltiples filas en un solo valor. Son el equivalente a las funciones de resumen de Excel (PROMEDIO, CONTAR, SUMA, etc.).

| Función SQL | Equivalente Excel | Descripción |
|-----------|-------------------|------------|
| `COUNT(*)` | `CONTARA()` | Cuenta filas |
| `AVG(col)` | `PROMEDIO()` | Promedio |
| `SUM(col)` | `SUMA()` | Suma total |
| `MIN(col)` | `MIN()` | Valor mínimo |
| `MAX(col)` | `MAX()` | Valor máximo |

`GROUP BY` agrupa las filas antes de aplicar la función. Es como crear una **tabla dinámica** en Excel.

In [ ]:
# ============================================================
# Ejemplo 6: Promedio de calificaciones por carrera
# ============================================================
# Equivalente a una tabla dinámica con carrera en filas
# y promedio como valor (función: PROMEDIO)

resultado = consulta("""
SELECT carrera,
       COUNT(*) AS total_estudiantes,
       ROUND(AVG(promedio), 2) AS promedio_general,
       ROUND(MIN(promedio), 2) AS promedio_minimo,
       ROUND(MAX(promedio), 2) AS promedio_maximo
FROM rendimiento
GROUP BY carrera
ORDER BY promedio_general DESC
""")

print("🏫 Rendimiento académico por carrera:")
print(resultado.to_string(index=False))

In [ ]:
# ============================================================
# Ejemplo 7: Conteo de estudiantes por género y carrera
# ============================================================

resultado = consulta("""
SELECT carrera, genero,
       COUNT(*) AS cantidad
FROM rendimiento
GROUP BY carrera, genero
ORDER BY carrera, genero
""")

print("👥 Distribución por carrera y género:")
print(resultado.to_string(index=False))

In [ ]:
# ============================================================
# Ejemplo 8: Estadísticas de asistencia por semestre
# ============================================================

resultado = consulta("""
SELECT semestre,
       COUNT(*) AS estudiantes,
       ROUND(AVG(asistencia_pct), 1) AS asistencia_promedio,
       ROUND(AVG(promedio), 2) AS promedio_notas
FROM rendimiento
GROUP BY semestre
ORDER BY semestre
""")

print("📅 Indicadores por semestre:")
print(resultado.to_string(index=False))

In [ ]:
# ============================================================
# Ejemplo 9: HAVING — filtrar DESPUÉS de agrupar
# ============================================================
# HAVING es como WHERE, pero se aplica sobre los grupos.
# Pregunta: ¿Qué carreras tienen más de 700 estudiantes?

resultado = consulta("""
SELECT carrera,
       COUNT(*) AS total_estudiantes,
       ROUND(AVG(promedio), 2) AS promedio_general
FROM rendimiento
GROUP BY carrera
HAVING COUNT(*) > 700
ORDER BY total_estudiantes DESC
""")

print("🏫 Carreras con más de 700 estudiantes:")
print(resultado.to_string(index=False))

### 💡 WHERE vs HAVING

Una confusión frecuente es cuándo usar `WHERE` y cuándo `HAVING`:

- **`WHERE`** filtra filas **antes** de agrupar. Ejemplo: "Solo considerar estudiantes de Economía"
- **`HAVING`** filtra grupos **después** de agrupar. Ejemplo: "Solo mostrar carreras con más de 700 estudiantes"

```sql
-- Orden de ejecución:
-- 1. FROM     → ¿De dónde?
-- 2. WHERE    → ¿Qué filas incluir?
-- 3. GROUP BY → ¿Cómo agrupar?
-- 4. HAVING   → ¿Qué grupos incluir?
-- 5. SELECT   → ¿Qué columnas mostrar?
-- 6. ORDER BY → ¿En qué orden?
-- 7. LIMIT    → ¿Cuántos resultados?
```

---

## 🔗 Sección 5: JOIN — Combinando tablas

En el mundo real, los datos están distribuidos en **múltiples tablas**. Por ejemplo:
- Una tabla con datos de **matrícula** (período, tipo de ingreso, estatus)
- Otra tabla con datos de **rendimiento** (notas, asistencia, materias)

Para combinarlas usamos `JOIN`, que conecta filas de dos tablas a través de una columna en común.

### Tipos de JOIN

| Tipo | Resultado |
|------|----------|
| `INNER JOIN` | Solo filas que tienen coincidencia en **ambas** tablas |
| `LEFT JOIN` | **Todas** las filas de la tabla izquierda + coincidencias de la derecha |
| `RIGHT JOIN` | **Todas** las filas de la tabla derecha + coincidencias de la izquierda |
| `FULL OUTER JOIN` | **Todas** las filas de ambas tablas |

In [ ]:
# ============================================================
# Primero, exploremos ambas tablas para encontrar columnas comunes
# ============================================================

print("📝 Columnas en 'rendimiento':")
print(f"   {list(df_rendimiento.columns)}")
print(f"   Carreras: {df_rendimiento['carrera'].unique().tolist()}")

print("\n📝 Columnas en 'matricula':")
print(f"   {list(df_matricula.columns)}")
print(f"   Carreras: {df_matricula['carrera'].unique().tolist()}")

print("\n✅ Columna en común: 'carrera' — la usaremos para el JOIN")

In [ ]:
# ============================================================
# Ejemplo 10: INNER JOIN — Matrícula + Rendimiento por carrera
# ============================================================
# Creamos un resumen de matrícula por carrera y lo unimos
# con un resumen de rendimiento por carrera.

resultado = consulta("""
SELECT 
    m.carrera,
    COUNT(DISTINCT m.periodo) AS periodos_disponibles,
    COUNT(m.rowid) AS total_registros_matricula,
    r.total_estudiantes,
    r.promedio_general
FROM matricula m
INNER JOIN (
    SELECT carrera,
           COUNT(*) AS total_estudiantes,
           ROUND(AVG(promedio), 2) AS promedio_general
    FROM rendimiento
    GROUP BY carrera
) r ON m.carrera = r.carrera
GROUP BY m.carrera
ORDER BY m.carrera
""")

print("🔗 Matrícula + Rendimiento combinados por carrera:")
print(resultado.to_string(index=False))

In [ ]:
# ============================================================
# Ejemplo 11: LEFT JOIN — Ver estatus de matrícula con datos
# de rendimiento (manteniendo todos los registros de matrícula)
# ============================================================

# Primero creemos un resumen de estatus por carrera desde matrícula
resultado = consulta("""
SELECT 
    m_resumen.carrera,
    m_resumen.estatus,
    m_resumen.cantidad_matricula,
    r_resumen.promedio_general
FROM (
    SELECT carrera, estatus, COUNT(*) AS cantidad_matricula
    FROM matricula
    GROUP BY carrera, estatus
) m_resumen
LEFT JOIN (
    SELECT carrera, ROUND(AVG(promedio), 2) AS promedio_general
    FROM rendimiento
    GROUP BY carrera
) r_resumen ON m_resumen.carrera = r_resumen.carrera
ORDER BY m_resumen.carrera, m_resumen.estatus
""")

print("🔗 LEFT JOIN: Matrícula por estatus con promedio de rendimiento:")
print(resultado.to_string(index=False))
print("\n📌 Con LEFT JOIN, mantenemos todos los registros de la tabla izquierda (matrícula)")
print("   incluso si no hubiera coincidencia en la tabla derecha (rendimiento).")

In [ ]:
# ============================================================
# Ejemplo 12: JOIN práctico — Tasa de deserción + rendimiento
# ============================================================
# Combinamos la tasa de retiro por carrera con el rendimiento
# promedio para buscar correlaciones.

resultado = consulta("""
SELECT 
    t_desercion.carrera,
    t_desercion.total_registros,
    t_desercion.retirados,
    ROUND(t_desercion.retirados * 100.0 / t_desercion.total_registros, 1) AS pct_retirados,
    t_rendimiento.promedio_general,
    t_rendimiento.asistencia_promedio
FROM (
    SELECT carrera, 
           COUNT(*) AS total_registros,
           SUM(CASE WHEN estatus = 'Retirado' THEN 1 ELSE 0 END) AS retirados
    FROM matricula
    GROUP BY carrera
) t_desercion
INNER JOIN (
    SELECT carrera,
           ROUND(AVG(promedio), 2) AS promedio_general,
           ROUND(AVG(asistencia_pct), 1) AS asistencia_promedio
    FROM rendimiento
    GROUP BY carrera
) t_rendimiento ON t_desercion.carrera = t_rendimiento.carrera
ORDER BY pct_retirados DESC
""")

print("🔗 Relación entre deserción y rendimiento académico:")
print(resultado.to_string(index=False))
print("\n📊 ¿Se observa alguna relación entre menor promedio y mayor deserción?")

---

## 🧩 Sección 6: Subconsultas

Una **subconsulta** (o *subquery*) es una consulta dentro de otra consulta. Es útil cuando necesitas el resultado de un cálculo intermedio para filtrar o comparar.

**Analogía:** Es como usar el resultado de una fórmula dentro de otra fórmula en Excel. Por ejemplo, primero calculas el promedio general y luego filtras los que están por encima.

In [ ]:
# ============================================================
# Ejemplo 13: Estudiantes con promedio superior al promedio general
# ============================================================
# La subconsulta calcula el promedio general.
# La consulta principal filtra los estudiantes por encima de ese valor.

# Primero veamos cuál es el promedio general
promedio_gral = consulta("""
SELECT ROUND(AVG(promedio), 2) AS promedio_general
FROM rendimiento
""")
print(f"El promedio general es: {promedio_gral.iloc[0, 0]}")
print()

In [ ]:
# Ahora usemos una subconsulta para filtrar automáticamente
resultado = consulta("""
SELECT estudiante_id, carrera, promedio
FROM rendimiento
WHERE promedio > (
    SELECT AVG(promedio) FROM rendimiento
)
ORDER BY promedio DESC
LIMIT 15
""")

print(f"\n🌟 Estudiantes por encima del promedio general (top 15):")
print(resultado.to_string(index=False))

In [ ]:
# ============================================================
# Ejemplo 14: Carreras con promedio superior al promedio global
# ============================================================
# Subconsulta en HAVING: ¿qué carreras tienen un promedio
# superior al promedio global de toda la facultad?

resultado = consulta("""
SELECT carrera,
       COUNT(*) AS estudiantes,
       ROUND(AVG(promedio), 2) AS promedio_carrera
FROM rendimiento
GROUP BY carrera
HAVING AVG(promedio) > (
    SELECT AVG(promedio) FROM rendimiento
)
ORDER BY promedio_carrera DESC
""")

print("🏫 Carreras con promedio superior al promedio global de FACES:")
print(resultado.to_string(index=False))

In [ ]:
# ============================================================
# Ejemplo 15: Subconsulta con IN — Estudiantes en carreras
# donde la tasa de retiro supera el 25%
# ============================================================

resultado = consulta("""
SELECT carrera,
       COUNT(*) AS total,
       ROUND(AVG(promedio), 2) AS promedio
FROM rendimiento
WHERE carrera IN (
    SELECT carrera
    FROM matricula
    GROUP BY carrera
    HAVING SUM(CASE WHEN estatus = 'Retirado' THEN 1 ELSE 0 END) * 100.0 / COUNT(*) > 25
)
GROUP BY carrera
""")

print("⚠️ Rendimiento en carreras con tasa de retiro > 25%:")
print(resultado.to_string(index=False))

---

## ✏️ Ejercicios

Ahora es tu turno. Usa la función `consulta()` para escribir tus propias consultas SQL.

**Recuerda:**
- La tabla `rendimiento` tiene: `estudiante_id, carrera, semestre, genero, edad, promedio, materias_inscritas, materias_aprobadas, materias_reprobadas, asistencia_pct, beca, trabaja`
- La tabla `matricula` tiene: `periodo, carrera, genero, edad, tipo_ingreso, sede, estatus`

### Ejercicio 1: Top 10 estudiantes por carrera

Escribe una consulta que muestre los **10 estudiantes con mejor promedio** para **cada carrera**.

**Pista:** Puedes usar una subconsulta con `WHERE carrera = ...` y `LIMIT 10`, o usar `ROW_NUMBER()` si te sientes aventurero. También puedes hacer 4 consultas separadas (una por carrera) y eso está bien.

In [ ]:
# ============================================================
# Ejercicio 1: Top 10 estudiantes por carrera
# ============================================================

# Tu código aquí
# Opción A: Una consulta por carrera
# resultado = consulta("""
#     SELECT estudiante_id, carrera, promedio
#     FROM rendimiento
#     WHERE carrera = '???'
#     ORDER BY promedio DESC
#     LIMIT 10
# """)

# Opción B: Usando subconsultas (más avanzado)
# resultado = consulta("""
#     SELECT ...
# """)


### Ejercicio 2: Tasa de deserción por carrera

Usando la tabla `matricula`, calcula la **tasa de deserción** (porcentaje de estudiantes con estatus 'Retirado') para cada carrera.

**Pista:** Usa `SUM(CASE WHEN estatus = 'Retirado' THEN 1 ELSE 0 END)` para contar retirados, y divídelo entre `COUNT(*)` para obtener el porcentaje.

In [ ]:
# ============================================================
# Ejercicio 2: Tasa de deserción por carrera
# ============================================================

# Tu código aquí
# resultado = consulta("""
#     SELECT carrera,
#            COUNT(*) AS total,
#            SUM(CASE WHEN estatus = '???' THEN 1 ELSE 0 END) AS retirados,
#            ROUND(??? * 100.0 / ???, 1) AS tasa_desercion_pct
#     FROM matricula
#     GROUP BY carrera
#     ORDER BY tasa_desercion_pct DESC
# """)


### Ejercicio 3: JOIN — Promedio por tipo de ingreso

Combina las tablas `matricula` y `rendimiento` (usando `carrera` como columna común) para calcular:
- El número de registros por `tipo_ingreso` 
- El promedio general de rendimiento asociado a cada tipo de ingreso

**Pista:** Crea un resumen de rendimiento por carrera y únelo con un resumen de matrícula por tipo_ingreso y carrera.

In [ ]:
# ============================================================
# Ejercicio 3: Promedio por tipo de ingreso (JOIN)
# ============================================================

# Tu código aquí
# resultado = consulta("""
#     SELECT ...
#     FROM ... 
#     JOIN ...
#     ON ...
#     GROUP BY ...
# """)


---

## 📋 Resumen

En esta sesión aprendimos:

| Concepto | Descripción | Ejemplo |
|----------|-----------|--------|
| `SELECT ... FROM` | Seleccionar columnas de una tabla | `SELECT carrera, promedio FROM rendimiento` |
| `WHERE` | Filtrar filas por condiciones | `WHERE promedio > 15` |
| `AND / OR` | Combinar condiciones | `WHERE carrera = 'Economia' AND beca = 'True'` |
| `ORDER BY` | Ordenar resultados | `ORDER BY promedio DESC` |
| `LIMIT` | Limitar cantidad de resultados | `LIMIT 10` |
| `COUNT, AVG, SUM, MIN, MAX` | Funciones de agregación | `SELECT AVG(promedio) FROM rendimiento` |
| `GROUP BY` | Agrupar filas para agregar | `GROUP BY carrera` |
| `HAVING` | Filtrar grupos (post-agregación) | `HAVING COUNT(*) > 700` |
| `INNER JOIN` | Combinar tablas (solo coincidencias) | `FROM t1 INNER JOIN t2 ON t1.col = t2.col` |
| `LEFT JOIN` | Combinar tablas (todos de la izquierda) | `FROM t1 LEFT JOIN t2 ON t1.col = t2.col` |
| Subconsultas | Consulta dentro de otra consulta | `WHERE promedio > (SELECT AVG(promedio) ...)` |

### SQL Cheat Sheet

```sql
-- Consulta básica
SELECT columna1, columna2 FROM tabla WHERE condicion;

-- Agregación
SELECT columna, COUNT(*), AVG(valor) FROM tabla GROUP BY columna;

-- JOIN
SELECT a.col1, b.col2 FROM tabla_a a INNER JOIN tabla_b b ON a.clave = b.clave;

-- Subconsulta
SELECT * FROM tabla WHERE valor > (SELECT AVG(valor) FROM tabla);
```

## 📚 Referencias

### Recursos recomendados

1. **SQLBolt** — Tutorial interactivo gratuito de SQL: [https://sqlbolt.com/](https://sqlbolt.com/)
2. **W3Schools SQL Tutorial** — Referencia rápida con ejemplos: [https://www.w3schools.com/sql/](https://www.w3schools.com/sql/)
3. **Mode Analytics SQL Tutorial** — SQL orientado a análisis de datos: [https://mode.com/sql-tutorial/](https://mode.com/sql-tutorial/)
4. **SQLite Documentation** — Documentación oficial de SQLite: [https://www.sqlite.org/docs.html](https://www.sqlite.org/docs.html)
5. **Khan Academy — Intro to SQL** — Curso gratuito en video: [https://www.khanacademy.org/computing/computer-programming/sql](https://www.khanacademy.org/computing/computer-programming/sql)

### Plataformas para practicar

- **SQLZoo** — Ejercicios interactivos: [https://sqlzoo.net/](https://sqlzoo.net/)
- **HackerRank SQL** — Desafíos de SQL por nivel: [https://www.hackerrank.com/domains/sql](https://www.hackerrank.com/domains/sql)
- **LeetCode Database** — Problemas SQL: [https://leetcode.com/problemset/database/](https://leetcode.com/problemset/database/)

### Bibliografía

- Beaulieu, A. (2020). *Learning SQL* (3rd ed.). O'Reilly Media.
- Forta, B. (2019). *SQL in 10 Minutes a Day* (5th ed.). Sams Publishing.

---

*Notebook desarrollado para el programa de Formación Docente en Ciencia de Datos y Business Intelligence — FACES, Universidad Catolica Andres Bello (UCAB).*